# Fixed V2 — 512×128 Transformer vs RAG-Transformer

This revision keeps the same 512×128 OTFS system assumptions and fixes the main bottleneck observed in the previous run: **Doppler/support recovery**.

Main changes:
- empirical temporal support prior learned only from the training split;
- local Doppler attention + cross-path attention across the 9 physical paths;
- stronger support/ranking loss;
- validation-selected score fusion (no test-set tuning);
- structured physics IHT refinement instead of one-shot argmax support selection;
- identical physics refinement for Transformer and RAG-Transformer.

**Important:** the paper does not disclose the exact large-grid sensing matrix/tap locations. The 64-measurement sensing operator remains an explicit implementation assumption.


## Runtime fix (V3)

The previous run failed on T4 with `c10::Half` overflow because the ranking loss used `masked_fill(..., -1e9)` while AMP had converted the logits to FP16. This revision:

- keeps Transformer forward propagation under AMP;
- computes the complete loss in FP32;
- replaces the unsafe FP16 mask value `-1e9` with `-1e4`;
- checks for non-finite loss before backward propagation.

No system/model-performance result has been changed or fabricated; the notebook must be re-run to obtain new results.


## V4 refinement — validation-only high-SNR physics polish

The previous executed run reached **RAG-Transformer NMSE = -18.138 dB at 12 dB**, only about 0.26 dB from the base-paper numerical reference (-18.395 dB).

This revision does **not** alter the 512×128 grid, 20 pilots, 64-measurement assumption, channel bank, or test labels. It adds two validation-selected refinements:

1. SNR-banded regularization for the 9-coefficient LS fit (high-SNR estimates should not be over-regularized).
2. Residual-guided coordinate support polish over a small candidate list formed from model score + matched correlation + neighboring Doppler bins.

All solver choices are selected on the validation split and frozen before test evaluation. There is no clipping or test-set tuning to force a result below the paper reference.


In [ ]:

# STEP 1 — Imports and configuration
import os, gc, time, math, random, copy, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 20260709
QUICK_MODE = False

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)

OUT = Path("/kaggle/working/otfs_512x128_rag_tf") if Path("/kaggle/working").exists() else Path.cwd() / "otfs_512x128_rag_tf"
OUT.mkdir(parents=True, exist_ok=True)

# ---------------- Paper-disclosed system ----------------
N_DOPPLER = 512
M_DELAY = 128
GRID_SIZE = N_DOPPLER * M_DELAY

FC_HZ = 28e9
SCS_HZ = 15e3
SPEED_KMH = 120.0
PAPER_MAX_DOPPLER_HZ = 1112.0

NUM_USERS = 100 if not QUICK_MODE else 20
NUM_PATHS = 9
MAX_DELAY_S = 5e-6
MODULATION = "16-QAM"
NUM_PILOTS = 20

# Paper does not disclose exact receiver sensing dimension.
# 64 spectral CS measurements are an explicit implementation assumption.
NUM_MEASUREMENTS = 64

EBN0_DB = np.array([0, 2, 4, 6, 8, 10, 12], dtype=np.float32)

# ---------------- Temporal dataset for retrieval ----------------
TRAIN_STEPS_PER_USER = 80 if not QUICK_MODE else 16
VAL_STEPS_PER_USER = 10 if not QUICK_MODE else 4
TEST_STEPS_PER_USER = 10 if not QUICK_MODE else 4
TOTAL_STEPS_PER_USER = TRAIN_STEPS_PER_USER + VAL_STEPS_PER_USER + TEST_STEPS_PER_USER

CHANNEL_RHO = 0.995
DOPPLER_STAY_PROB = 0.86

# ---------------- RAG ----------------
TOP_K = 6

# ---------------- Transformer ----------------
D_MODEL = 96
N_HEADS = 6
N_LAYERS = 2
PATH_LAYERS = 1
DROPOUT = 0.05

BATCH_SIZE = 64 if not QUICK_MODE else 32
MAX_EPOCHS = 40 if not QUICK_MODE else 5
PATIENCE = 8
MODEL_SEEDS = [42, 2026, 3407] if not QUICK_MODE else [42]

PAPER_NMSE_12_LINEAR = 0.01447
PAPER_NMSE_12_DB = 10*np.log10(PAPER_NMSE_12_LINEAR)

print(f"OTFS grid = {N_DOPPLER} × {M_DELAY} = {GRID_SIZE:,} DD elements")
print(f"Users = {NUM_USERS}, paths = {NUM_PATHS}, pilots = {NUM_PILOTS}")
print(f"Paper max Doppler = ±{PAPER_MAX_DOPPLER_HZ:.0f} Hz")
print(f"Base-paper quoted NMSE @12 dB = {PAPER_NMSE_12_LINEAR:.5f} = {PAPER_NMSE_12_DB:.2f} dB")


In [ ]:

# STEP 2 — 512×128 delay/Doppler geometry

DELAY_RESOLUTION_S = 1.0 / (M_DELAY * SCS_HZ)
DOPPLER_RESOLUTION_HZ = SCS_HZ / N_DOPPLER

# Paper: 9 taps, max excess delay 5 μs.
# Exact nine tap positions are not disclosed in the simulation section.
PATH_DELAYS_S = np.linspace(0.0, MAX_DELAY_S, NUM_PATHS)
PATH_DELAY_BINS_FLOAT = PATH_DELAYS_S / DELAY_RESOLUTION_S

MAX_DOPPLER_BIN = int(np.round(PAPER_MAX_DOPPLER_HZ / DOPPLER_RESOLUTION_HZ))
DOPPLER_BINS = np.arange(-MAX_DOPPLER_BIN, MAX_DOPPLER_BIN + 1, dtype=np.int32)
DOPPLER_HZ = DOPPLER_BINS * DOPPLER_RESOLUTION_HZ

Q = len(DOPPLER_BINS)
K = NUM_PATHS * Q

print(f"Delay resolution = {DELAY_RESOLUTION_S*1e6:.6f} μs")
print("Assumed 9 delay taps [μs]:", np.round(PATH_DELAYS_S*1e6, 4))
print(f"Doppler resolution = {DOPPLER_RESOLUTION_HZ:.6f} Hz")
print(f"Doppler candidate bins = {DOPPLER_BINS[0]} ... {DOPPLER_BINS[-1]}")
print(f"Doppler hypotheses/path = {Q}")
print(f"Structured sparse dictionary = {NUM_PATHS} × {Q} = {K} atoms")



## STEP 3 — Full-grid pilot sensing operator

Let \(P[k,l]\) be the 20-pilot DD grid. Under the ideal circular DD shift model, a path with fractional delay-bin shift \(d\) and Doppler-bin shift \(q\) produces a shifted pilot response.

Instead of allocating a 65,536×65,536 shift matrix, the 2-D DFT shift theorem is used:

\[
\mathcal F_2\{P[k-q,l-d]\}[u,v]
=
\mathcal F_2\{P\}[u,v]
e^{-j2\pi(uq/N+vd/M)}.
\]

Sampling 64 Fourier coordinates gives a dense, memory-safe compressive sensing matrix \(A\in\mathbb C^{64\times K}\).


In [ ]:

# STEP 3 — Build a memory-safe 512×128 physical pilot sensing matrix

rng = np.random.default_rng(SEED)

pilot_grid = np.zeros((N_DOPPLER, M_DELAY), dtype=np.complex64)
pilot_flat_idx = rng.choice(GRID_SIZE, NUM_PILOTS, replace=False)
pilot_qpsk = (
    (2*rng.integers(0,2,NUM_PILOTS)-1)
    + 1j*(2*rng.integers(0,2,NUM_PILOTS)-1)
) / np.sqrt(2.0)
pilot_grid.reshape(-1)[pilot_flat_idx] = pilot_qpsk.astype(np.complex64)

pilot_spectrum = np.fft.fft2(pilot_grid, norm="ortho")

def build_A_from_measurement_coords(coords):
    u = coords[:,0].astype(np.float64)
    v = coords[:,1].astype(np.float64)
    amp = pilot_spectrum[coords[:,0], coords[:,1]].astype(np.complex128)

    cols = []
    for d_float in PATH_DELAY_BINS_FLOAT:
        for q_bin in DOPPLER_BINS:
            phase = np.exp(
                -1j*2*np.pi*(
                    u*float(q_bin)/N_DOPPLER
                    + v*float(d_float)/M_DELAY
                )
            )
            cols.append(amp*phase)

    A_local = np.stack(cols, axis=1).astype(np.complex64)
    A_local /= np.linalg.norm(A_local, axis=0, keepdims=True) + 1e-12
    return A_local

def mutual_coherence(A_local):
    G = np.abs(A_local.conj().T @ A_local)
    np.fill_diagonal(G, 0.0)
    return float(np.max(G))

search_trials = 8 if not QUICK_MODE else 2
best = None

valid = np.argwhere(np.abs(pilot_spectrum) > np.percentile(np.abs(pilot_spectrum), 25))

for trial in range(search_trials):
    pick = rng.choice(len(valid), NUM_MEASUREMENTS, replace=False)
    coords = valid[pick]
    A_try = build_A_from_measurement_coords(coords)
    mu = mutual_coherence(A_try)

    if best is None or mu < best[0]:
        best = (mu, coords.copy(), A_try.copy())

A_COHERENCE, MEASUREMENT_COORDS, A = best
A_RANK = np.linalg.matrix_rank(A)

print("A shape:", A.shape)
print("A rank:", A_RANK, "/", NUM_MEASUREMENTS)
print("Mutual coherence:", A_COHERENCE)
print("20 transmitted pilot symbols are preserved.")



## STEP 4 — 100-user temporal Rayleigh channel bank

The paper specifies independent Rayleigh taps but does not define a temporal correlation law for retrieval.

For the proposed RAG experiment, each user is simulated as a **causal channel trajectory**:

- each of the 9 paths has one active Doppler hypothesis;
- Doppler support performs a slow random walk;
- complex path gain follows an AR(1) Rayleigh process;
- training comes first in time, then validation, then test.

This temporal assumption is part of **our proposed RAG setup**, not a claim about an undisclosed base-paper parameter.


In [ ]:

# STEP 4 — Generate user channel trajectories

def sample_next_doppler_indices(prev, rg):
    out = prev.copy()
    for g in range(NUM_PATHS):
        if rg.random() > DOPPLER_STAY_PROB:
            step = -1 if rg.random() < 0.5 else 1
            out[g] = int(np.clip(out[g] + step, 0, Q-1))
    return out

def generate_user_trajectory(rg):
    supports = np.zeros((TOTAL_STEPS_PER_USER, NUM_PATHS), dtype=np.int16)
    gains = np.zeros((TOTAL_STEPS_PER_USER, NUM_PATHS), dtype=np.complex64)

    supports[0] = rg.choice(Q, NUM_PATHS, replace=False)

    g0 = (rg.normal(size=NUM_PATHS) + 1j*rg.normal(size=NUM_PATHS)) / np.sqrt(2*NUM_PATHS)
    gains[0] = g0.astype(np.complex64)

    sigma_innov = np.sqrt(max(1.0-CHANNEL_RHO**2, 1e-8))

    for t in range(1, TOTAL_STEPS_PER_USER):
        supports[t] = sample_next_doppler_indices(supports[t-1], rg)

        innovation = (
            rg.normal(size=NUM_PATHS) + 1j*rg.normal(size=NUM_PATHS)
        ) / np.sqrt(2*NUM_PATHS)

        gains[t] = (
            CHANNEL_RHO*gains[t-1]
            + sigma_innov*innovation
        ).astype(np.complex64)

    return supports, gains

def structured_h_from_support_gain(supports, gains):
    n = len(supports)
    H = np.zeros((n, NUM_PATHS, Q), dtype=np.complex64)
    rows = np.arange(n)[:,None]
    groups = np.arange(NUM_PATHS)[None,:]
    H[rows, groups, supports] = gains
    return H

all_support = []
all_gain = []
all_user = []
all_time = []
all_split = []

rg_channels = np.random.default_rng(SEED + 100)

for u in range(NUM_USERS):
    s, g = generate_user_trajectory(rg_channels)
    all_support.append(s)
    all_gain.append(g)
    all_user.extend([u]*TOTAL_STEPS_PER_USER)
    all_time.extend(range(TOTAL_STEPS_PER_USER))
    all_split.extend(
        ["train"]*TRAIN_STEPS_PER_USER
        + ["val"]*VAL_STEPS_PER_USER
        + ["test"]*TEST_STEPS_PER_USER
    )

ALL_SUPPORT = np.concatenate(all_support, axis=0)
ALL_GAIN = np.concatenate(all_gain, axis=0)
ALL_USER = np.asarray(all_user, dtype=np.int16)
ALL_TIME = np.asarray(all_time, dtype=np.int16)
ALL_SPLIT = np.asarray(all_split)

ALL_H = structured_h_from_support_gain(ALL_SUPPORT, ALL_GAIN)
ALL_H_FLAT = ALL_H.reshape(len(ALL_H), K)

print("Total channel states:", len(ALL_H))
print("Train:", np.sum(ALL_SPLIT=="train"))
print("Val:", np.sum(ALL_SPLIT=="val"))
print("Test:", np.sum(ALL_SPLIT=="test"))
print("H shape:", ALL_H.shape)


In [ ]:

# STEP 5 — Observation generation and physical warm start

def add_measurement_noise(clean_y, snr_db, rg):
    clean_y = np.asarray(clean_y)
    sig_pow = np.mean(np.abs(clean_y)**2, axis=1, keepdims=True) + 1e-12
    snr_lin = 10.0**(np.asarray(snr_db).reshape(-1,1)/10.0)
    noise_var = sig_pow / snr_lin

    noise = np.sqrt(noise_var/2.0) * (
        rg.normal(size=clean_y.shape) + 1j*rg.normal(size=clean_y.shape)
    )
    return (clean_y + noise).astype(np.complex64), noise_var[:,0].astype(np.float32)

def group_physical_warmstart(Y, noise_var):
    # Choose one Doppler candidate per delay group using matched correlation,
    # then fit nine complex path coefficients by regularized LS.
    Y = np.asarray(Y, dtype=np.complex64)
    corr = (Y @ A.conj()).reshape(len(Y), NUM_PATHS, Q)

    H0 = np.zeros((len(Y), NUM_PATHS, Q), dtype=np.complex64)
    support0 = np.argmax(np.abs(corr), axis=2)

    for i in range(len(Y)):
        flat_support = np.arange(NUM_PATHS)*Q + support0[i]
        As = A[:, flat_support]

        lam = float(noise_var[i]) * NUM_PATHS
        gram = As.conj().T@As + lam*np.eye(NUM_PATHS)
        rhs = As.conj().T@Y[i]
        coef = np.linalg.solve(gram, rhs)

        H0[i, np.arange(NUM_PATHS), support0[i]] = coef.astype(np.complex64)

    return H0, corr.astype(np.complex64), support0

def make_train_or_val_bank(split, seed):
    idx = np.where(ALL_SPLIT==split)[0]
    H = ALL_H[idx]
    Hf = H.reshape(len(H), K)

    rg = np.random.default_rng(seed)
    snr_db = rg.uniform(0.0, 12.0, size=len(idx)).astype(np.float32)

    clean = (Hf @ A.T).astype(np.complex64)
    Y, nv = add_measurement_noise(clean, snr_db, rg)
    H0, corr, support0 = group_physical_warmstart(Y, nv)

    return {
        "idx": idx,
        "H": H,
        "Y": Y,
        "NV": nv,
        "SNR": snr_db,
        "H0": H0,
        "CORR": corr,
        "SUPPORT0": support0,
        "USER": ALL_USER[idx],
        "TIME": ALL_TIME[idx],
        "TRUE_SUPPORT": ALL_SUPPORT[idx],
    }

train_bank = make_train_or_val_bank("train", SEED+200)
val_bank = make_train_or_val_bank("val", SEED+300)

print("Train Y:", train_bank["Y"].shape)
print("Train H:", train_bank["H"].shape)



## STEP 6 — Leakage-safe RAG memory

Each training state acts as a memory item:

\[
(\text{observable correlation key},\ \text{historical CSI support/magnitude}).
\]

For a query:

- only memory from the **same user** is searched;
- a training query can retrieve only **earlier training times**;
- validation/test queries can retrieve only that user's **training split**;
- current validation/test CSI is never stored in memory.

The retrieval key uses normalized matched-filter magnitudes, so it does not depend on random complex channel phase.


In [ ]:

# STEP 6 — Leakage-safe similarity retrieval + empirical temporal prior

def corr_key(corr):
    x = np.abs(corr).reshape(len(corr), K).astype(np.float32)
    x /= np.linalg.norm(x, axis=1, keepdims=True) + 1e-8
    return x

TRAIN_KEYS = corr_key(train_bank["CORR"])

USER_TRAIN_POS = {}
for u in range(NUM_USERS):
    USER_TRAIN_POS[u] = np.where(train_bank["USER"] == u)[0]

def retrieve_priors(bank, training_query=False):
    """Similarity RAG. Memory always comes from the training split."""
    query_keys = corr_key(bank["CORR"])

    prior_support = np.zeros((len(query_keys), NUM_PATHS, Q), dtype=np.float32)
    prior_mag = np.zeros_like(prior_support)

    for i in range(len(query_keys)):
        u = int(bank["USER"][i])
        t = int(bank["TIME"][i])
        cand = USER_TRAIN_POS[u]

        if training_query:
            cand = cand[train_bank["TIME"][cand] < t]

        if len(cand) == 0:
            continue

        sims = TRAIN_KEYS[cand] @ query_keys[i]
        k = min(TOP_K, len(cand))
        local = np.argpartition(sims, -k)[-k:]
        chosen = cand[local]
        chosen_sims = sims[local]

        logits = 12.0 * (chosen_sims - np.max(chosen_sims))
        w = np.exp(logits)
        w /= np.sum(w) + 1e-12

        Hmem = train_bank["H"][chosen]
        prior_support[i] = np.sum(
            (np.abs(Hmem) > 1e-9).astype(np.float32) * w[:, None, None],
            axis=0
        )
        prior_mag[i] = np.sum(
            np.abs(Hmem).astype(np.float32) * w[:, None, None],
            axis=0
        )

    return prior_support, prior_mag

# Learn a Q×Q Doppler transition matrix from TRAINING trajectories only.
TRANSITION_COUNTS = np.full((Q, Q), 1e-3, dtype=np.float64)

for u in range(NUM_USERS):
    pos = USER_TRAIN_POS[u]
    pos = pos[np.argsort(train_bank["TIME"][pos])]
    s = train_bank["TRUE_SUPPORT"][pos]
    tt = train_bank["TIME"][pos]

    for j in range(1, len(pos)):
        if int(tt[j]) != int(tt[j-1]) + 1:
            continue
        for g in range(NUM_PATHS):
            TRANSITION_COUNTS[int(s[j-1, g]), int(s[j, g])] += 1.0

TRANSITION = TRANSITION_COUNTS / TRANSITION_COUNTS.sum(axis=1, keepdims=True)
_TRANSITION_POWER_CACHE = {0: np.eye(Q, dtype=np.float64), 1: TRANSITION.copy()}

def transition_power(dt):
    dt = int(max(dt, 0))
    if dt not in _TRANSITION_POWER_CACHE:
        _TRANSITION_POWER_CACHE[dt] = np.linalg.matrix_power(TRANSITION, dt)
    return _TRANSITION_POWER_CACHE[dt]

def temporal_prior_from_train(bank, training_query=False):
    """
    Causal temporal prior.
    Uses only a historical TRAINING support state from the same user and an
    empirical transition matrix estimated from training trajectories.
    """
    out = np.zeros((len(bank["Y"]), NUM_PATHS, Q), dtype=np.float32)

    for i in range(len(bank["Y"])):
        u = int(bank["USER"][i])
        t = int(bank["TIME"][i])
        cand = USER_TRAIN_POS[u]

        if training_query:
            cand = cand[train_bank["TIME"][cand] < t]
        else:
            cand = cand[train_bank["TIME"][cand] < t]

        if len(cand) == 0:
            continue

        times = train_bank["TIME"][cand]
        j = cand[np.argmax(times)]
        t0 = int(train_bank["TIME"][j])
        dt = max(t - t0, 1)
        T = transition_power(dt)

        previous_support = train_bank["TRUE_SUPPORT"][j]
        for g in range(NUM_PATHS):
            out[i, g] = T[int(previous_support[g])].astype(np.float32)

    return out

train_prior_support, train_prior_mag = retrieve_priors(train_bank, training_query=True)
val_prior_support, val_prior_mag = retrieve_priors(val_bank, training_query=False)

train_temporal_prior = temporal_prior_from_train(train_bank, training_query=True)
val_temporal_prior = temporal_prior_from_train(val_bank, training_query=False)

print("RAG priors ready.")
print("Mean max train similarity-support prior:", float(train_prior_support.max(axis=2).mean()))
print("Mean max val similarity-support prior:", float(val_prior_support.max(axis=2).mean()))
print("Mean max val temporal prior:", float(val_temporal_prior.max(axis=2).mean()))


In [ ]:

# STEP 7 — Token construction

def build_features(bank, prior_support, prior_mag, temporal_prior):
    H0 = bank["H0"]
    corr = bank["CORR"]

    corr_scale = np.max(np.abs(corr), axis=2, keepdims=True) + 1e-8
    corr_n = corr / corr_scale

    prior_mag_n = prior_mag / (np.max(prior_mag, axis=2, keepdims=True) + 1e-8)

    doppler_norm = np.broadcast_to(
        (DOPPLER_HZ / PAPER_MAX_DOPPLER_HZ)[None, None, :],
        (len(H0), NUM_PATHS, Q)
    ).astype(np.float32)

    delay_norm = np.broadcast_to(
        (PATH_DELAYS_S / MAX_DELAY_S)[None, :, None],
        (len(H0), NUM_PATHS, Q)
    ).astype(np.float32)

    snr_norm = np.broadcast_to(
        (bank["SNR"] / 12.0)[:, None, None],
        (len(H0), NUM_PATHS, Q)
    ).astype(np.float32)

    # Columns 4:7 are the RAG-only features.
    tokens = np.stack(
        [
            H0.real,
            H0.imag,
            corr_n.real,
            corr_n.imag,
            prior_support,
            prior_mag_n,
            temporal_prior,
            doppler_norm,
            delay_norm,
            snr_norm,
        ],
        axis=-1
    ).astype(np.float32)

    Y = bank["Y"]
    y_scale = np.sqrt(np.mean(np.abs(Y)**2, axis=1, keepdims=True)) + 1e-8
    Yn = Y / y_scale

    global_feat = np.concatenate(
        [
            Yn.real,
            Yn.imag,
            (bank["SNR"] / 12.0)[:, None],
        ],
        axis=1
    ).astype(np.float32)

    target = np.stack([bank["H"].real, bank["H"].imag], axis=-1).astype(np.float32)
    support_target = bank["TRUE_SUPPORT"].astype(np.int64)

    return tokens, global_feat, target, support_target

train_tokens, train_global, train_target, train_support_target = build_features(
    train_bank, train_prior_support, train_prior_mag, train_temporal_prior
)
val_tokens, val_global, val_target, val_support_target = build_features(
    val_bank, val_prior_support, val_prior_mag, val_temporal_prior
)

TOKEN_DIM = train_tokens.shape[-1]
GLOBAL_DIM = train_global.shape[-1]

print("Token shape:", train_tokens.shape)
print("Global shape:", train_global.shape)
print("Target shape:", train_target.shape)


In [ ]:

# STEP 8 — Dataset / loaders

class CEDataset(Dataset):
    def __init__(self, tokens, global_feat, target, support_target, Y):
        self.tokens = torch.from_numpy(tokens)
        self.global_feat = torch.from_numpy(global_feat)
        self.target = torch.from_numpy(target)
        self.support_target = torch.from_numpy(support_target)
        self.Y = torch.from_numpy(Y)

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, i):
        return {
            "tokens": self.tokens[i],
            "global": self.global_feat[i],
            "target": self.target[i],
            "support_target": self.support_target[i],
            "y": self.Y[i],
        }

train_ds = CEDataset(
    train_tokens, train_global, train_target, train_support_target, train_bank["Y"]
)
val_ds = CEDataset(
    val_tokens, val_global, val_target, val_support_target, val_bank["Y"]
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=False
)
val_loader = DataLoader(
    val_ds, batch_size=max(BATCH_SIZE,128), shuffle=False,
    num_workers=2, pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


## STEP 9 — Two-stage structured Transformer

The previous model attended only within each Doppler sequence. This revision adds a second attention stage across the 9 physical delay paths.

1. **Local Doppler encoder:** attention over the 77 Doppler hypotheses for each path.
2. **Cross-path encoder:** pools each path, attends across all 9 paths, and broadcasts the resulting context back to each candidate.

The same architecture is used by both models. The plain Transformer zeros the three RAG-only token channels; the RAG-Transformer keeps them.


In [ ]:

# STEP 9 — Local Doppler attention + cross-path attention

class GroupedChannelTransformer(nn.Module):
    def __init__(self, use_rag):
        super().__init__()
        self.use_rag = bool(use_rag)

        self.token_proj = nn.Linear(TOKEN_DIM, D_MODEL)
        self.global_proj = nn.Sequential(
            nn.Linear(GLOBAL_DIM, D_MODEL),
            nn.GELU(),
            nn.Linear(D_MODEL, D_MODEL)
        )

        self.doppler_pos = nn.Parameter(torch.randn(1, 1, Q, D_MODEL) * 0.02)
        self.delay_embed = nn.Parameter(torch.randn(1, NUM_PATHS, 1, D_MODEL) * 0.02)

        local_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=4 * D_MODEL,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.local_encoder = nn.TransformerEncoder(local_layer, num_layers=N_LAYERS)

        self.pool_gate = nn.Linear(D_MODEL, 1)

        path_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=2 * D_MODEL,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.path_encoder = nn.TransformerEncoder(path_layer, num_layers=PATH_LAYERS)

        self.norm = nn.LayerNorm(D_MODEL)
        self.support_head = nn.Linear(D_MODEL, 1)
        self.residual_head = nn.Linear(D_MODEL, 2)

        # Begin close to the physical warm start.
        nn.init.zeros_(self.residual_head.weight)
        nn.init.zeros_(self.residual_head.bias)

    def forward(self, tokens, global_feat):
        x = tokens.clone()

        if not self.use_rag:
            # similarity support, magnitude prior, temporal prior
            x[..., 4:7] = 0.0

        B = x.shape[0]

        z = self.token_proj(x)
        z = z + self.doppler_pos + self.delay_embed
        z = z + self.global_proj(global_feat)[:, None, None, :]

        # Stage 1: local attention across 77 Doppler hypotheses.
        z = z.reshape(B * NUM_PATHS, Q, D_MODEL)
        z = self.local_encoder(z)
        z = z.reshape(B, NUM_PATHS, Q, D_MODEL)

        # Stage 2: summarize each physical path and attend across 9 paths.
        pool_w = torch.softmax(self.pool_gate(z).squeeze(-1), dim=2)
        path_summary = torch.sum(pool_w.unsqueeze(-1) * z, dim=2)
        path_context = self.path_encoder(path_summary)

        z = z + path_context[:, :, None, :]
        z = self.norm(z)

        support_logits = self.support_head(z).squeeze(-1)
        residual = self.residual_head(z)

        pred = tokens[..., 0:2] + residual
        return pred, support_logits

tmp = GroupedChannelTransformer(use_rag=True).to(DEVICE)
print("Parameters:", sum(p.numel() for p in tmp.parameters()) / 1e6, "M")
del tmp


In [ ]:

# STEP 10 — Support-focused coefficient + ranking + physics loss

A_TORCH = torch.from_numpy(A.astype(np.complex64)).to(DEVICE)

def ri_to_complex(x):
    return torch.complex(x[..., 0], x[..., 1])

def model_loss(pred_ri, logits, batch, epoch):
    # IMPORTANT: compute the loss in FP32 even when the model forward uses AMP/FP16.
    # This prevents masked_fill / ranking constants and complex physics operations
    # from overflowing half precision on T4.
    pred_ri = pred_ri.float()
    logits = logits.float()
    target_ri = batch["target"].float()
    support_target = batch["support_target"].long()

    # One true Doppler hypothesis per physical path.
    support_loss = F.cross_entropy(
        logits.reshape(-1, Q),
        support_target.reshape(-1),
        label_smoothing=0.01
    )

    # Hard-negative ranking term: true support should beat the strongest false hypothesis.
    true_logit = torch.gather(logits, 2, support_target.unsqueeze(-1)).squeeze(-1)
    support_mask = F.one_hot(support_target, num_classes=Q).bool()
    hard_negative = logits.masked_fill(support_mask, -1e4).max(dim=2).values
    rank_loss = F.relu(1.0 - true_logit + hard_negative).mean()

    pred = ri_to_complex(pred_ri)
    target = ri_to_complex(target_ri)

    gather_idx = support_target[..., None, None].expand(-1, -1, 1, 2)
    pred_active_ri = torch.gather(pred_ri, 2, gather_idx).squeeze(2)
    target_active_ri = torch.gather(target_ri, 2, gather_idx).squeeze(2)

    active_num = torch.sum((pred_active_ri - target_active_ri)**2, dim=(1, 2))
    active_den = torch.sum(target_active_ri**2, dim=(1, 2)) + 1e-8
    active_coef_loss = torch.mean(torch.log1p(20.0 * active_num / active_den))

    # Suppress coefficient energy on inactive hypotheses.
    inactive_mask = (~support_mask).float()
    inactive_energy = torch.sum(
        inactive_mask * torch.sum(pred_ri**2, dim=-1), dim=(1, 2)
    )
    target_energy = torch.sum(target_active_ri**2, dim=(1, 2)) + 1e-8
    inactive_loss = torch.mean(inactive_energy / target_energy)

    # Low-weight differentiable physics consistency.
    prob = torch.softmax(logits, dim=-1)
    pred_soft = pred * prob
    yhat = pred_soft.reshape(pred.shape[0], K) @ A_TORCH.T
    y = batch["y"]
    physics_loss = torch.mean(
        torch.sum(torch.abs(yhat - y)**2, dim=1)
        / (torch.sum(torch.abs(y)**2, dim=1) + 1e-8)
    )

    physics_w = 0.0 if epoch <= 5 else 0.02

    total = (
        1.00 * support_loss
        + 0.20 * rank_loss
        + 0.25 * active_coef_loss
        + 0.02 * inactive_loss
        + physics_w * physics_loss
    )
    return total

@torch.no_grad()
def eval_model_metrics(model, loader):
    model.eval()
    nsum = 0.0
    dsum = 0.0
    correct = 0
    total = 0

    for batch in loader:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        pred_ri, logits = model(batch["tokens"], batch["global"])
        pred = ri_to_complex(pred_ri)
        target = ri_to_complex(batch["target"])

        nsum += torch.sum(torch.abs(pred - target)**2).item()
        dsum += torch.sum(torch.abs(target)**2).item()

        pred_support = torch.argmax(logits, dim=2)
        correct += torch.sum(pred_support == batch["support_target"]).item()
        total += batch["support_target"].numel()

    raw_db = 10 * np.log10(nsum / (dsum + 1e-12) + 1e-15)
    path_acc = correct / max(total, 1)
    return float(raw_db), float(path_acc)


In [ ]:

# STEP 11 — Train both models with identical seeds/hyperparameters

def train_one(seed, use_rag):
    seed_all(seed)

    model = GroupedChannelTransformer(use_rag=use_rag).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=MAX_EPOCHS, eta_min=2e-5
    )
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_acc = -1.0
    best_nmse = float("inf")
    best_state = None
    bad = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []

        for batch in train_loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            opt.zero_grad(set_to_none=True)

            # AMP is used for the Transformer forward only.
            # Loss is intentionally evaluated in FP32 for numerical stability.
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=torch.cuda.is_available()
            ):
                pred_ri, logits = model(batch["tokens"], batch["global"])

            with torch.autocast(
                device_type="cuda",
                enabled=False
            ):
                loss = model_loss(pred_ri, logits, batch, epoch)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss detected: seed={seed}, epoch={epoch}. "
                    "Check the printed batch/model values before continuing."
                )

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.item()))

        sched.step()
        val_db, val_acc = eval_model_metrics(model, val_loader)
        history.append((epoch, float(np.mean(losses)), val_db, val_acc))

        name = "RAG-TF" if use_rag else "TF"
        print(
            f"{name} seed={seed} epoch={epoch:02d} "
            f"loss={np.mean(losses):.4f} "
            f"val_raw_NMSE={val_db:.3f} dB "
            f"val_path_acc={val_acc:.4f}"
        )

        improved = (val_acc > best_acc + 1e-4) or (
            abs(val_acc - best_acc) <= 1e-4 and val_db < best_nmse - 0.01
        )

        if improved:
            best_acc = val_acc
            best_nmse = val_db
            best_state = copy.deepcopy(
                {k: v.detach().cpu() for k, v in model.state_dict().items()}
            )
            bad = 0
        else:
            bad += 1

        if epoch >= 12 and bad >= PATIENCE:
            print("Early stop.")
            break

    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, history, best_nmse, best_acc

transformer_models = []
rag_models = []

for seed in MODEL_SEEDS:
    model, hist, val_nmse, val_acc = train_one(seed, use_rag=False)
    transformer_models.append(model)
    torch.save(model.state_dict(), OUT / f"transformer_seed{seed}.pt")
    print(f"Transformer seed {seed}: val path acc={val_acc:.4f}, raw NMSE={val_nmse:.3f} dB")

for seed in MODEL_SEEDS:
    model, hist, val_nmse, val_acc = train_one(seed, use_rag=True)
    rag_models.append(model)
    torch.save(model.state_dict(), OUT / f"rag_transformer_seed{seed}.pt")
    print(f"RAG-Transformer seed {seed}: val path acc={val_acc:.4f}, raw NMSE={val_nmse:.3f} dB")


In [ ]:

# STEP 12 — Ensemble inference + validation-only solver calibration

@torch.no_grad()
def ensemble_predict(models, tokens, global_feat):
    pred_list = []
    prob_list = []
    bs = 256 if not QUICK_MODE else 64

    for model in models:
        model.eval()
        pred_chunks = []
        prob_chunks = []

        for s in range(0, len(tokens), bs):
            t = torch.from_numpy(tokens[s:s+bs]).to(DEVICE)
            g = torch.from_numpy(global_feat[s:s+bs]).to(DEVICE)

            p, logits = model(t, g)
            pred_chunks.append(p.float().cpu().numpy())
            prob_chunks.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())

        pred_list.append(np.concatenate(pred_chunks, axis=0))
        prob_list.append(np.concatenate(prob_chunks, axis=0))

    pred_ri = np.mean(np.stack(pred_list, axis=0), axis=0)
    prob = np.mean(np.stack(prob_list, axis=0), axis=0)

    pred_c = pred_ri[..., 0] + 1j * pred_ri[..., 1]
    pred_mag = np.abs(pred_c)
    pred_mag /= np.max(pred_mag, axis=2, keepdims=True) + 1e-8

    return pred_c.astype(np.complex64), prob.astype(np.float32), pred_mag.astype(np.float32)

def normalized_corr_mag(bank):
    x = np.abs(bank["CORR"]).astype(np.float32)
    return x / (np.max(x, axis=2, keepdims=True) + 1e-8)

def support_metrics(true_support, pred_support):
    exact = np.mean(np.all(true_support == pred_support, axis=1))
    path_acc = np.mean(true_support == pred_support)
    return float(exact), float(path_acc)

def nmse_linear(H_true, H_est):
    num = np.sum(np.abs(H_true - H_est)**2, axis=(1, 2))
    den = np.sum(np.abs(H_true)**2, axis=(1, 2)) + 1e-12
    return float(np.mean(num / den))

# ------------------------------------------------------------------
# Validation ensemble predictions
# ------------------------------------------------------------------
_, val_tf_prob, val_tf_mag = ensemble_predict(transformer_models, val_tokens, val_global)
_, val_rag_prob, val_rag_mag = ensemble_predict(rag_models, val_tokens, val_global)
val_corr_mag = normalized_corr_mag(val_bank)

# ------------------------------------------------------------------
# Validation-only score fusion
# ------------------------------------------------------------------
def calibrate_tf_weights(prob, mag, corr, truth, mask=None):
    if mask is None:
        mask = np.ones(len(prob), dtype=bool)

    best = None
    for wp in np.arange(0.5, 0.91, 0.1):
        for wm in np.arange(0.0, 0.31, 0.1):
            wc = 1.0 - wp - wm
            if wc < -1e-9 or wc > 0.4:
                continue
            score = wp*prob + wm*mag + wc*corr
            pred = np.argmax(score[mask], axis=2)
            acc = np.mean(pred == truth[mask])
            if best is None or acc > best[0]:
                best = (float(acc), float(wp), float(wm), float(wc))
    return best

def calibrate_rag_weights(prob, mag, corr, psup, temporal, truth, mask=None):
    if mask is None:
        mask = np.ones(len(prob), dtype=bool)

    best = None
    for wp in np.arange(0.3, 0.71, 0.1):
        for wm in (0.0, 0.1):
            for wr in (0.0, 0.1, 0.2):
                for wt in (0.0, 0.1, 0.2, 0.3, 0.4):
                    wc = 1.0 - wp - wm - wr - wt
                    if wc < -1e-9 or wc > 0.4:
                        continue
                    score = wp*prob + wm*mag + wc*corr + wr*psup + wt*temporal
                    pred = np.argmax(score[mask], axis=2)
                    acc = np.mean(pred == truth[mask])
                    if best is None or acc > best[0]:
                        best = (
                            float(acc), float(wp), float(wm),
                            float(wc), float(wr), float(wt)
                        )
    return best

# Use a dedicated high-SNR validation calibration for 8/10/12 dB testing.
VAL_HIGH = val_bank["SNR"] >= 8.0
VAL_LOW = ~VAL_HIGH

TF_FUSION_LOW = calibrate_tf_weights(
    val_tf_prob, val_tf_mag, val_corr_mag, val_support_target, VAL_LOW
)
TF_FUSION_HIGH = calibrate_tf_weights(
    val_tf_prob, val_tf_mag, val_corr_mag, val_support_target, VAL_HIGH
)

RAG_FUSION_LOW = calibrate_rag_weights(
    val_rag_prob, val_rag_mag, val_corr_mag,
    val_prior_support, val_temporal_prior, val_support_target, VAL_LOW
)
RAG_FUSION_HIGH = calibrate_rag_weights(
    val_rag_prob, val_rag_mag, val_corr_mag,
    val_prior_support, val_temporal_prior, val_support_target, VAL_HIGH
)

print("TF fusion LOW :", TF_FUSION_LOW)
print("TF fusion HIGH:", TF_FUSION_HIGH)
print("RAG fusion LOW :", RAG_FUSION_LOW)
print("RAG fusion HIGH:", RAG_FUSION_HIGH)

def fuse_tf_score(prob, mag, corr, high_snr):
    _, wp, wm, wc = TF_FUSION_HIGH if high_snr else TF_FUSION_LOW
    return (wp*prob + wm*mag + wc*corr).astype(np.float32)

def fuse_rag_score(prob, mag, corr, psup, temporal, high_snr):
    _, wp, wm, wc, wr, wt = RAG_FUSION_HIGH if high_snr else RAG_FUSION_LOW
    return (wp*prob + wm*mag + wc*corr + wr*psup + wt*temporal).astype(np.float32)

# ------------------------------------------------------------------
# Structured IHT warm polish
# ------------------------------------------------------------------
def structured_iht_refine(
    Y, noise_var, score,
    step=1.2, iters=8, score_bias=0.15, reg_scale=1.0
):
    n = len(Y)
    support = np.argmax(score, axis=2).astype(np.int64)
    Hout = np.zeros((n, NUM_PATHS, Q), dtype=np.complex64)
    eye9 = np.eye(NUM_PATHS, dtype=np.complex64)
    score_n = score / (np.max(score, axis=2, keepdims=True) + 1e-8)

    for i in range(n):
        supp = support[i].copy()
        h = np.zeros(K, dtype=np.complex64)
        lam = float(reg_scale) * float(noise_var[i]) * NUM_PATHS

        flat = np.arange(NUM_PATHS)*Q + supp
        As = A[:, flat]
        coef = np.linalg.solve(
            As.conj().T @ As + lam*eye9,
            As.conj().T @ Y[i]
        )
        h[flat] = coef.astype(np.complex64)

        for _ in range(iters):
            residual = Y[i] - A @ h
            z = h + float(step) * (A.conj().T @ residual)

            zg = np.abs(z.reshape(NUM_PATHS, Q))
            zg /= np.max(zg, axis=1, keepdims=True) + 1e-8
            metric = zg + float(score_bias) * score_n[i]

            supp = np.argmax(metric, axis=1)
            flat = np.arange(NUM_PATHS)*Q + supp
            As = A[:, flat]

            coef = np.linalg.solve(
                As.conj().T @ As + lam*eye9,
                As.conj().T @ Y[i]
            )
            h.fill(0.0)
            h[flat] = coef.astype(np.complex64)

        support[i] = supp
        Hout[i, np.arange(NUM_PATHS), supp] = coef.astype(np.complex64)

    return Hout, support

# ------------------------------------------------------------------
# Residual-guided local coordinate support search
# ------------------------------------------------------------------
def residual_coordinate_polish(
    Y, noise_var, score, corr_mag, init_support,
    top_score=4, top_corr=3, neighbor_radius=1,
    passes=2, prior_penalty=0.002, reg_scale=0.5
):
    """
    Validation-selected, target-free local MAP/physics search.

    For each physical path, candidate Dopplers come from:
      * current support,
      * top model-score hypotheses,
      * top matched-correlation hypotheses,
      * neighboring Doppler bins.

    Each candidate is judged by regularized LS data residual plus a
    small model-score prior penalty. No true/test CSI is used.
    """
    n = len(Y)
    eye9 = np.eye(NUM_PATHS, dtype=np.complex64)
    out = np.zeros((n, NUM_PATHS, Q), dtype=np.complex64)
    final_support = init_support.copy().astype(np.int64)

    score_n = score / (np.max(score, axis=2, keepdims=True) + 1e-8)

    for i in range(n):
        supp = final_support[i].copy()
        y = Y[i]
        ynorm = float(np.vdot(y, y).real) + 1e-12
        lam = float(reg_scale) * float(noise_var[i]) * NUM_PATHS

        candidate_lists = []
        for g in range(NUM_PATHS):
            cand = set([int(supp[g])])

            k1 = min(int(top_score), Q)
            k2 = min(int(top_corr), Q)

            cand.update(
                np.argpartition(score[i, g], -k1)[-k1:].astype(int).tolist()
            )
            cand.update(
                np.argpartition(corr_mag[i, g], -k2)[-k2:].astype(int).tolist()
            )

            for d in range(-int(neighbor_radius), int(neighbor_radius)+1):
                q = int(supp[g]) + d
                if 0 <= q < Q:
                    cand.add(q)

            candidate_lists.append(sorted(cand))

        for _ in range(int(passes)):
            changed = False

            for g in range(NUM_PATHS):
                best_q = int(supp[g])
                best_obj = float("inf")

                for q in candidate_lists[g]:
                    trial = supp.copy()
                    trial[g] = int(q)

                    flat = np.arange(NUM_PATHS)*Q + trial
                    As = A[:, flat]
                    coef = np.linalg.solve(
                        As.conj().T @ As + lam*eye9,
                        As.conj().T @ y
                    )
                    residual = y - As @ coef
                    data_term = float(np.vdot(residual, residual).real) / ynorm

                    # Smaller penalty for hypotheses already favored by the model.
                    prior_term = float(prior_penalty) * (1.0 - float(score_n[i, g, q]))
                    obj = data_term + prior_term

                    if obj < best_obj:
                        best_obj = obj
                        best_q = int(q)

                if best_q != int(supp[g]):
                    supp[g] = best_q
                    changed = True

            if not changed:
                break

        flat = np.arange(NUM_PATHS)*Q + supp
        As = A[:, flat]
        coef = np.linalg.solve(
            As.conj().T @ As + lam*eye9,
            As.conj().T @ y
        )

        final_support[i] = supp
        out[i, np.arange(NUM_PATHS), supp] = coef.astype(np.complex64)

    return out, final_support

# ------------------------------------------------------------------
# Validation-only solver selection
# ------------------------------------------------------------------
# Build low/high validation scores.
val_tf_score_low = fuse_tf_score(val_tf_prob, val_tf_mag, val_corr_mag, high_snr=False)
val_tf_score_high = fuse_tf_score(val_tf_prob, val_tf_mag, val_corr_mag, high_snr=True)
val_rag_score_low = fuse_rag_score(
    val_rag_prob, val_rag_mag, val_corr_mag,
    val_prior_support, val_temporal_prior, high_snr=False
)
val_rag_score_high = fuse_rag_score(
    val_rag_prob, val_rag_mag, val_corr_mag,
    val_prior_support, val_temporal_prior, high_snr=True
)

# First select IHT parameters on a modest validation subset.
IHT_GRID = [
    (1.0, 0.10), (1.2, 0.10), (1.4, 0.10),
    (1.0, 0.20), (1.2, 0.20), (1.4, 0.20),
]
IHT_CAL_N = min(220, len(val_bank["Y"]))

best_iht = None
for step, bias in IHT_GRID:
    Htf, _ = structured_iht_refine(
        val_bank["Y"][:IHT_CAL_N], val_bank["NV"][:IHT_CAL_N],
        val_tf_score_low[:IHT_CAL_N],
        step=step, iters=8, score_bias=bias, reg_scale=1.0
    )
    Hrg, _ = structured_iht_refine(
        val_bank["Y"][:IHT_CAL_N], val_bank["NV"][:IHT_CAL_N],
        val_rag_score_low[:IHT_CAL_N],
        step=step, iters=8, score_bias=bias, reg_scale=1.0
    )

    avg_nmse = 0.5 * (
        nmse_linear(val_bank["H"][:IHT_CAL_N], Htf)
        + nmse_linear(val_bank["H"][:IHT_CAL_N], Hrg)
    )

    if best_iht is None or avg_nmse < best_iht[0]:
        best_iht = (float(avg_nmse), float(step), float(bias))

_, PHYS_STEP, PHYS_SCORE_BIAS = best_iht
PHYS_ITERS = 8

# High-SNR validation subset for the final polish.
high_idx = np.where(VAL_HIGH)[0]
if len(high_idx) < 60:
    high_idx = np.argsort(val_bank["SNR"])[-min(250, len(val_bank["SNR"])):]
else:
    high_idx = high_idx[:min(250, len(high_idx))]

# Get IHT supports once; then search only lightweight local-polish settings.
Htf0, stf0 = structured_iht_refine(
    val_bank["Y"][high_idx], val_bank["NV"][high_idx],
    val_tf_score_high[high_idx],
    step=PHYS_STEP, iters=PHYS_ITERS, score_bias=PHYS_SCORE_BIAS, reg_scale=1.0
)
Hrg0, srg0 = structured_iht_refine(
    val_bank["Y"][high_idx], val_bank["NV"][high_idx],
    val_rag_score_high[high_idx],
    step=PHYS_STEP, iters=PHYS_ITERS, score_bias=PHYS_SCORE_BIAS, reg_scale=1.0
)

POLISH_GRID = [
    # top_score, top_corr, passes, prior_penalty, reg_scale
    (3, 2, 1, 0.000, 0.25),
    (4, 3, 1, 0.001, 0.25),
    (4, 3, 2, 0.002, 0.25),
    (4, 3, 2, 0.002, 0.50),
    (5, 3, 2, 0.003, 0.50),
]

best_polish = None
for ts, tc, ps, pp, rs in POLISH_GRID:
    Htf, _ = residual_coordinate_polish(
        val_bank["Y"][high_idx], val_bank["NV"][high_idx],
        val_tf_score_high[high_idx], val_corr_mag[high_idx], stf0,
        top_score=ts, top_corr=tc, passes=ps,
        prior_penalty=pp, reg_scale=rs
    )
    Hrg, _ = residual_coordinate_polish(
        val_bank["Y"][high_idx], val_bank["NV"][high_idx],
        val_rag_score_high[high_idx], val_corr_mag[high_idx], srg0,
        top_score=ts, top_corr=tc, passes=ps,
        prior_penalty=pp, reg_scale=rs
    )

    avg_nmse = 0.5 * (
        nmse_linear(val_bank["H"][high_idx], Htf)
        + nmse_linear(val_bank["H"][high_idx], Hrg)
    )

    if best_polish is None or avg_nmse < best_polish[0]:
        best_polish = (
            float(avg_nmse), int(ts), int(tc), int(ps),
            float(pp), float(rs)
        )

_, POLISH_TOP_SCORE, POLISH_TOP_CORR, POLISH_PASSES, POLISH_PRIOR, HIGH_REG_SCALE = best_polish

print(
    "Validation-selected IHT:",
    dict(step=PHYS_STEP, score_bias=PHYS_SCORE_BIAS, iters=PHYS_ITERS)
)
print(
    "Validation-selected HIGH-SNR coordinate polish:",
    dict(
        top_score=POLISH_TOP_SCORE,
        top_corr=POLISH_TOP_CORR,
        passes=POLISH_PASSES,
        prior_penalty=POLISH_PRIOR,
        reg_scale=HIGH_REG_SCALE,
    )
)


In [ ]:

# STEP 13 — Fixed-SNR test: Transformer vs RAG-Transformer only

test_idx = np.where(ALL_SPLIT == "test")[0]
H_test = ALL_H[test_idx]
Hf_test = H_test.reshape(len(H_test), K)
USER_test = ALL_USER[test_idx]
TIME_test = ALL_TIME[test_idx]
SUPPORT_test = ALL_SUPPORT[test_idx]

results = []

for snr_db in EBN0_DB:
    rg = np.random.default_rng(SEED + 1000 + int(snr_db*10))

    clean = (Hf_test @ A.T).astype(np.complex64)
    snr_vec = np.full(len(H_test), float(snr_db), dtype=np.float32)
    Y, nv = add_measurement_noise(clean, snr_vec, rg)

    H0, corr, support0 = group_physical_warmstart(Y, nv)

    bank = {
        "idx": test_idx,
        "H": H_test,
        "Y": Y,
        "NV": nv,
        "SNR": snr_vec,
        "H0": H0,
        "CORR": corr,
        "SUPPORT0": support0,
        "USER": USER_test,
        "TIME": TIME_test,
        "TRUE_SUPPORT": SUPPORT_test,
    }

    p_sup, p_mag = retrieve_priors(bank, training_query=False)
    p_temp = temporal_prior_from_train(bank, training_query=False)

    z_sup = np.zeros_like(p_sup)
    z_mag = np.zeros_like(p_mag)
    z_temp = np.zeros_like(p_temp)

    tf_tokens, tf_global, _, _ = build_features(bank, z_sup, z_mag, z_temp)
    rag_tokens, rag_global, _, _ = build_features(bank, p_sup, p_mag, p_temp)

    _, tf_prob, tf_mag = ensemble_predict(transformer_models, tf_tokens, tf_global)
    _, rag_prob, rag_mag = ensemble_predict(rag_models, rag_tokens, rag_global)

    corr_mag = normalized_corr_mag(bank)
    high_snr = bool(float(snr_db) >= 8.0)

    tf_score = fuse_tf_score(tf_prob, tf_mag, corr_mag, high_snr=high_snr)
    rag_score = fuse_rag_score(
        rag_prob, rag_mag, corr_mag, p_sup, p_temp, high_snr=high_snr
    )

    # Stage A: shared structured IHT.
    H_tf_iht, s_tf_iht = structured_iht_refine(
        Y, nv, tf_score,
        step=PHYS_STEP, iters=PHYS_ITERS,
        score_bias=PHYS_SCORE_BIAS, reg_scale=1.0
    )
    H_rag_iht, s_rag_iht = structured_iht_refine(
        Y, nv, rag_score,
        step=PHYS_STEP, iters=PHYS_ITERS,
        score_bias=PHYS_SCORE_BIAS, reg_scale=1.0
    )

    if high_snr:
        # Stage B: validation-selected local residual/MAP polish.
        H_tf, s_tf = residual_coordinate_polish(
            Y, nv, tf_score, corr_mag, s_tf_iht,
            top_score=POLISH_TOP_SCORE,
            top_corr=POLISH_TOP_CORR,
            passes=POLISH_PASSES,
            prior_penalty=POLISH_PRIOR,
            reg_scale=HIGH_REG_SCALE
        )
        H_rag, s_rag = residual_coordinate_polish(
            Y, nv, rag_score, corr_mag, s_rag_iht,
            top_score=POLISH_TOP_SCORE,
            top_corr=POLISH_TOP_CORR,
            passes=POLISH_PASSES,
            prior_penalty=POLISH_PRIOR,
            reg_scale=HIGH_REG_SCALE
        )
    else:
        H_tf, s_tf = H_tf_iht, s_tf_iht
        H_rag, s_rag = H_rag_iht, s_rag_iht

    tf_nmse = nmse_linear(H_test, H_tf)
    rag_nmse = nmse_linear(H_test, H_rag)

    tf_db = 10*np.log10(tf_nmse + 1e-15)
    rag_db = 10*np.log10(rag_nmse + 1e-15)

    tf_exact, tf_path = support_metrics(SUPPORT_test, s_tf)
    rag_exact, rag_path = support_metrics(SUPPORT_test, s_rag)

    results.append({
        "EbN0_dB": float(snr_db),
        "Transformer_NMSE": tf_nmse,
        "RAG_Transformer_NMSE": rag_nmse,
        "Transformer_NMSE_dB": tf_db,
        "RAG_Transformer_NMSE_dB": rag_db,
        "RAG_Gain_dB": tf_db - rag_db,
        "Transformer_ExactSupport": tf_exact,
        "RAG_ExactSupport": rag_exact,
        "Transformer_PathAccuracy": tf_path,
        "RAG_PathAccuracy": rag_path,
    })

    print(
        f"{snr_db:>4.0f} dB | "
        f"TF={tf_db:>7.3f} dB | "
        f"RAG-TF={rag_db:>7.3f} dB | "
        f"gain={tf_db-rag_db:+.3f} dB | "
        f"path acc TF/RAG={tf_path:.3f}/{rag_path:.3f} | "
        f"exact={tf_exact:.3f}/{rag_exact:.3f}"
    )

results_df = pd.DataFrame(results).set_index("EbN0_dB")
display(results_df)


In [ ]:

# STEP 14 — Final plots / 12-dB summary / saved CSV

results_df.to_csv(OUT/"Transformer_vs_RAG_512x128_results.csv")

tf12 = float(results_df.loc[12.0,"Transformer_NMSE_dB"])
rag12 = float(results_df.loc[12.0,"RAG_Transformer_NMSE_dB"])
gain12 = float(results_df.loc[12.0,"RAG_Gain_dB"])

print("========== 512×128 FINAL SUMMARY @ 12 dB ==========")
print(f"Transformer:      {tf12:.3f} dB")
print(f"RAG-Transformer:  {rag12:.3f} dB")
print(f"RAG gain:         {gain12:+.3f} dB")
print(f"Base-paper quoted numerical reference: {PAPER_NMSE_12_DB:.3f} dB")
print(f"Gap to paper reference (positive = our RAG is better): {PAPER_NMSE_12_DB-rag12:+.3f} dB")

if gain12 > 0:
    print("PASS: retrieval improves the Transformer at 12 dB.")
else:
    print("This run does not show a positive RAG gain at 12 dB.")

if rag12 < PAPER_NMSE_12_DB:
    print("RAG-Transformer is numerically below the quoted paper reference in this implementation.")
else:
    print("RAG-Transformer remains above the quoted paper reference in this implementation.")

print(
    "\nImportant: the base-paper value is a numerical reference only. "
    "The paper does not disclose the exact 512×128 sensing matrix/noise normalization, "
    "so this notebook should not claim an exact apples-to-apples reproduction."
)

fig, ax = plt.subplots(figsize=(8.5,5.5))
ax.plot(results_df.index,results_df["Transformer_NMSE_dB"],marker="o",label="Transformer")
ax.plot(results_df.index,results_df["RAG_Transformer_NMSE_dB"],marker="o",label="RAG-Transformer")
ax.axhline(PAPER_NMSE_12_DB,linestyle="--",label="Base-paper quoted 12-dB NMSE")
ax.set_xlabel(r"$E_b/N_0$ (dB)")
ax.set_ylabel("NMSE (dB)")
ax.set_title("512×128 OTFS: Transformer vs RAG-Transformer")
ax.grid(True,alpha=0.4)
ax.legend()
fig.tight_layout()
fig.savefig(OUT/"NMSE_512x128_Transformer_vs_RAG.png",dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(8.5,5.5))
ax.plot(results_df.index,results_df["Transformer_ExactSupport"],marker="o",label="Transformer")
ax.plot(results_df.index,results_df["RAG_ExactSupport"],marker="o",label="RAG-Transformer")
ax.set_xlabel(r"$E_b/N_0$ (dB)")
ax.set_ylabel("Exact 9-path support accuracy")
ax.set_ylim(0,1.02)
ax.set_title("512×128 OTFS: Support Recovery")
ax.grid(True,alpha=0.4)
ax.legend()
fig.tight_layout()
fig.savefig(OUT/"Support_512x128_Transformer_vs_RAG.png",dpi=300)
plt.show()

print("Saved to:", OUT.resolve())


# What this fixed notebook establishes

This experiment remains a **paper-aligned 512×128 implementation**, not a bit-for-bit reproduction of undisclosed simulation internals.

The final comparison remains only **Transformer vs RAG-Transformer**. Both models use the same two-stage Transformer, training seeds, sensing operator, and structured physics refinement. The controlled difference is the RAG information (similarity-based historical support/magnitude plus a training-only empirical temporal prior).

Fusion weights and the physics-refinement step are selected on the validation split and frozen before test evaluation. Test labels are not used for tuning.
